# Module 20 — Multimodal Agents

> **SDKs:** `google-genai` (simulated), `pydantic`

| Part | Topic |
|------|-------|
| **1** | Vision & Video — processing frames and grounding |
| **2** | Computer Use & UI — navigating bounding boxes |
| **3** | Multimodal Tool Calling — audio as input |


---
## Part 1 — Vision & Video Processing

Agents can now process visual data natively without relying on OCR pipelines. In video analysis, temporal context allows agents to detect sequences of events.

In [ ]:
from dataclasses import dataclass
from typing import Literal

@dataclass
class VideoFrame:
    timestamp_s: float
    description: str  # Simulated image embedding

class MultimodalAnalyzer:
    def analyze_sequence(self, frames: list[VideoFrame], query: str) -> str:
        print(f"  [Analyzer] Processing {len(frames)} frames for query: '{query}'")
        found_events = []
        for f in frames:
            if "error" in f.description.lower() or "crash" in f.description.lower():
                found_events.append(f)
        
        if found_events:
            ev = found_events[0]
            return f"Event detected at {ev.timestamp_s}s: {ev.description}"
        return "No relevant events detected."

frames = [
    VideoFrame(0.0, "User opens application homepage"),
    VideoFrame(2.5, "User clicks 'Checkout' button"),
    VideoFrame(3.1, "Screen goes white, loading spinner appears"),
    VideoFrame(8.4, "Browser displays '500 Server Error' text in red"),
]

analyzer = MultimodalAnalyzer()
print("👁️  Video Sequence Analysis Demo")
print("=" * 60)
result = analyzer.analyze_sequence(frames, "Find when the checkout process failed")
print(f"\n  Result: {result}")


👁️  Video Sequence Analysis Demo
  [Analyzer] Processing 4 frames for query: 'Find when the checkout process failed'

  Result: Event detected at 8.4s: Browser displays '500 Server Error' text in red


---
## Part 2 — Computer Use & UI Interaction

Agents interacting with UIs generate explicit coordinate clicks or use accessible DOM elements.

In [ ]:
from dataclasses import dataclass

@dataclass
class BoundingBox:
    x: int
    y: int
    width: int
    height: int

@dataclass
class UIElement:
    id: str
    type: str
    text: str
    box: BoundingBox

class UIAgent:
    def __init__(self, screen_elements: list[UIElement]):
        self.elements = screen_elements
        
    def find_and_click(self, instruction: str) -> str:
        target = "Submit" if "submit" in instruction.lower() else "Cancel"
        for el in self.elements:
            if target in el.text:
                center_x = el.box.x + el.box.width // 2
                center_y = el.box.y + el.box.height // 2
                return f"Mouse.click({center_x}, {center_y})  # Clicked {el.text!r}"
        return "Element not found"

screen = [
    UIElement("btn-1", "button", "Cancel", BoundingBox(100, 200, 80, 30)),
    UIElement("btn-2", "button", "Submit Order", BoundingBox(200, 200, 120, 30)),
]

agent = UIAgent(screen)
print("🖱️  Computer Use UI Demo")
print("=" * 60)
action = agent.find_and_click("Please submit my order now")
print(f"  Instruction: 'Please submit my order now'")
print(f"  Action Executed: {action}")


🖱️  Computer Use UI Demo
  Instruction: 'Please submit my order now'
  Action Executed: Mouse.click(260, 215)  # Clicked 'Submit Order'


---
## Part 3 — Multimodal Tool Calling

Passing audio natively alongside tool schemas allows an agent to directly extract information from phone calls.

In [ ]:
def process_audio_call(audio_transcript: str) -> dict:
    """Simulates passing an audio stream to a function-calling model."""
    print(f"  [Model] Processing native audio stream...")
    if "cancel my subscription" in audio_transcript:
        return {"tool_name": "cancel_subscription", "args": {"reason": "user_request"}}
    return {"tool_name": "escalate_to_human", "args": {}}

print("🎙️  Multimodal Tool Calling Demo")
print("=" * 60)
simulated_audio = "[Audio: Customer saying 'I need to cancel my subscription because it's too expensive']"
print(f"  Input: {simulated_audio}")
call = process_audio_call(simulated_audio)
print(f"\n  Tool Invoked: {call['tool_name']}")
print(f"  Arguments   : {call['args']}")


🎙️  Multimodal Tool Calling Demo
  Input: [Audio: Customer saying 'I need to cancel my subscription because it's too expensive']
  [Model] Processing native audio stream...

  Tool Invoked: cancel_subscription
  Arguments   : {'reason': 'user_request'}
